# Steps 1–2: Labels + CLIP embeddings (Colab)

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all.

Outputs are saved to `MyDrive/cifar10n-project/features/`. Download that folder into your repo's `features/` when done.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Mount Google Drive and set paths
Data goes on Colab's local disk (fast); only the small output files go to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUT = '/content/drive/MyDrive/cifar10n-project/features'
DATA_DIR = '/content/data'
os.makedirs(DRIVE_OUT, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
LABEL_VERSION = 'random_label1'

In [ ]:
!pip install -q open_clip_torch

## Step 1: Load CIFAR-10 + CIFAR-10N labels

In [ ]:
!wget -q -nc -P /content/data https://github.com/UCSC-REAL/cifar-10-100n/raw/main/data/CIFAR-10_human.pt

import numpy as np, torch, torchvision
train = torchvision.datasets.CIFAR10(DATA_DIR, train=True, download=True)
test  = torchvision.datasets.CIFAR10(DATA_DIR, train=False, download=True)

human = torch.load(f'{DATA_DIR}/CIFAR-10_human.pt', weights_only=False)
print('Label versions:', list(human.keys()))

y_clean = np.array(train.targets)
assert (np.array(human['clean_label']) == y_clean).all(), 'Label order mismatch!'
print('Order check passed')

for k in human:
    if k != 'clean_label':
        print(f'{k:15s} noise rate = {(np.array(human[k]) != y_clean).mean():.4f}')

y_noisy = np.array(human[LABEL_VERSION])
np.save(f'{DRIVE_OUT}/y_noisy.npy', y_noisy)
np.save(f'{DRIVE_OUT}/y_clean.npy', y_clean)
np.save(f'{DRIVE_OUT}/y_test.npy', np.array(test.targets))

## Step 2: CLIP ViT-B/32 embeddings (~5–10 min on a T4)

In [ ]:
import open_clip, torch.nn.functional as F
from torch.utils.data import DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
model = model.to(device).eval()

def embed(train_flag):
    ds = torchvision.datasets.CIFAR10(DATA_DIR, train=train_flag, transform=preprocess)
    loader = DataLoader(ds, batch_size=256, num_workers=2, shuffle=False)  # keep order!
    out = []
    with torch.no_grad(), torch.autocast('cuda'):
        for i, (x, _) in enumerate(loader):
            f = model.encode_image(x.to(device))
            out.append(F.normalize(f.float(), dim=-1).cpu())
            if i % 40 == 0: print(f'  batch {i}/{len(loader)}')
    return torch.cat(out).numpy()

X_train = embed(True);  np.save(f'{DRIVE_OUT}/X_train.npy', X_train)
X_test  = embed(False); np.save(f'{DRIVE_OUT}/X_test.npy',  X_test)
print(X_train.shape, X_test.shape)

## Sanity check
Norms should be 1.0. A quick logistic regression on clean labels should score roughly in the mid-90s % — if it's far lower, something is wrong with the embeddings or label order.

In [ ]:
from sklearn.linear_model import LogisticRegression
print('norms:', np.linalg.norm(X_train[:5], axis=1))
clf = LogisticRegression(max_iter=1000).fit(X_train[:10000], y_clean[:10000])
print('quick clean-label test acc:', clf.score(X_test, np.array(test.targets)))
!ls -lh {DRIVE_OUT}